# ED Pipeline v8 — v6 Tracker Schema + Hard Phase‑1 Gate (Kaggle-optimized)
This version searches **working dir** and **/kaggle/input/** for `ed_pipeline_v8.py`.
- Optional override: set `MANUAL_V8_PATH = '/kaggle/input/your-dataset/ed_pipeline_v8.py'` in the next cell.
- Phase‑1 must pass before Phase‑2 runs.


In [ ]:
# (Optional) Set a manual path if your module lives in an attached dataset
MANUAL_V8_PATH = ''  # e.g., '/kaggle/input/ed-pipeline-ds/ed_pipeline_v8.py'


In [ ]:
# === Cell 0: Robust canonical module bind (working dir + /kaggle/input, with manual override) ===
from pathlib import Path
import runpy, os
SEARCH_NAMES = ['ed_pipeline_v8.py', 'ed_pipeline_v8(2).py']
SEARCH_ROOTS = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle/input')]

def _pick(paths):
    # Prefer exact 'ed_pipeline_v8.py' and shorter paths; prefer working dir
    def score(p: Path):
        w = 0
        if p.name == 'ed_pipeline_v8.py': w += 10
        if str(p).startswith(str(Path.cwd())) or '/kaggle/working' in str(p): w += 5
        return (-w, len(str(p)))
    return sorted(paths, key=score)[0] if paths else None

candidates = []
manual = MANUAL_V8_PATH.strip() if 'MANUAL_V8_PATH' in globals() else ''
if not manual:
    manual = os.environ.get('V8_PATH','').strip()
if manual:
    p = Path(manual)
    if p.exists(): candidates.append(p)

for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for name in SEARCH_NAMES:
        for p in root.rglob(name):
            if p.is_file(): candidates.append(p)

found_path = _pick(candidates)

PHASE1_GATE = False
if not found_path:
    print('HARD GATE: canonical module not found. Expected one of:', SEARCH_NAMES)
    print('Searched roots:', [str(r) for r in SEARCH_ROOTS])
    print('Tip: set MANUAL_V8_PATH or env V8_PATH to the exact file path in your attached dataset.')
else:
    print('Found canonical module at:', found_path)
    mod = runpy.run_path(str(found_path))
    WorkflowState = mod.get('WorkflowState')
    TinyCritics = mod.get('TinyCritics')
    CONFIG = mod.get('CONFIG')
    PHASE1_GATE = all(v is not None for v in [WorkflowState, TinyCritics, CONFIG])
    print('Core symbols present:', PHASE1_GATE)
PHASE1_GATE


In [ ]:
# === Cell A: Back-compat for WorkflowState.update_state_from_event (no class edits) ===
if not PHASE1_GATE:
    raise SystemExit('HARD GATE: missing canonical module / core symbols. Fix before proceeding.')
_WS = globals().get('WorkflowState')
assert _WS is not None, 'WorkflowState must be loaded.'
if not hasattr(_WS, 'update_state_from_event'):
    def _update_state_from_event(self, event: dict):
        for candidate in ('apply_event','update_from_event','update_from_dict','update'):
            fn = getattr(self, candidate, None)
            if callable(fn): return fn(event)
        backing = getattr(self, 'state', None)
        if backing is None: backing = {}; setattr(self, 'state', backing)
        for k,v in (event or {}).items():
            parts = str(k).split('.')
            d = backing
            for p in parts[:-1]:
                if p not in d or not isinstance(d[p], dict): d[p] = {}
                d = d[p]
            d[parts[-1]] = v
        return self
    setattr(_WS, 'update_state_from_event', _update_state_from_event)
_=_WS(role='nurse').update_state_from_event({'age':60,'vitals.sbp':95}); print('OK: update_state_from_event present')


In [ ]:
# === Cell T(v6): tracker_core (v6 schema, all .loc, no side effects) ===
import sys, types, pandas as pd, numpy as np
from pathlib import Path
assert 'CONFIG' in globals(), 'CONFIG must be defined.'
def _cfg(CONFIG, key, default=None):
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default
def _ensure_parent(p: Path): p=Path(p); p.parent.mkdir(parents=True, exist_ok=True)
def _utcnow_iso(): return pd.Timestamp.utcnow().isoformat()
class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv = Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists():
            pd.DataFrame(columns=['equip_id','name','location','status','last_seen','battery','confidence']).to_csv(self.status_csv, index=False)
    def read(self) -> pd.DataFrame:
        try: df = pd.read_csv(self.status_csv)
        except Exception: df = pd.DataFrame(columns=['equip_id','name','location','status','last_seen','battery','confidence'])
        for c in ['equip_id','name','location','status','last_seen','battery','confidence']:
            if c not in df.columns: df[c] = np.nan
        df['equip_id'] = df['equip_id'].astype(str)
        return df[['equip_id','name','location','status','last_seen','battery','confidence']]
    def upsert(self, rec: dict) -> None:
        df = self.read(); eqid = str(rec.get('equip_id',''))
        if (df['equip_id'] == eqid).any():
            idx = df.index[df['equip_id'] == eqid][0]
            for k in ['name','location','status','last_seen','battery','confidence']:
                if k in df.columns:
                    df.loc[idx, k] = rec.get(k, df.loc[idx, k])
        else:
            df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)
        df.to_csv(self.status_csv, index=False)
class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv = Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists():
            pd.DataFrame(columns=['equip_id','from','to','ts']).to_csv(self.moves_csv, index=False)
    def append(self, equip_id: str, loc_from: str, loc_to: str, ts_iso: str) -> None:
        row = pd.DataFrame([{'equip_id': str(equip_id), 'from': loc_from, 'to': loc_to, 'ts': ts_iso}])
        try:
            prev = pd.read_csv(self.moves_csv); df = pd.concat([prev, row], ignore_index=True)
        except Exception:
            df = row
        df.to_csv(self.moves_csv, index=False)
    def read(self) -> pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=['equip_id','from','to','ts'])
class SOPRegistry:
    def __init__(self, sop_csv: Path):
        self.sop_csv = Path(sop_csv); _ensure_parent(self.sop_csv)
        if not self.sop_csv.exists():
            pd.DataFrame([
                {'sop_id':'sop-triage','title':'ED Triage','url':'about:blank','status':'active'},
                {'sop_id':'sop-ecg','title':'ECG Acquisition','url':'about:blank','status':'active'},
                {'sop_id':'sop-sepsis','title':'Sepsis Bundle','url':'about:blank','status':'active'},
            ]).to_csv(self.sop_csv, index=False)
    def read(self):
        try: return pd.read_csv(self.sop_csv)
        except Exception: return None
class QRService:
    def __init__(self, out_dir: Path):
        self.out_dir = Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self, payload: str) -> str:
        try:
            import qrcode
            img = qrcode.make(payload)
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.png"; img.save(p); return str(p)
        except Exception:
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.txt"; p.write_text(payload); return str(p)
    def decode(self, path: str):
        try:
            from PIL import Image; from pyzbar.pyzbar import decode as _decode
            res = _decode(Image.open(path));
            if res: return res[0].data.decode('utf-8', errors='ignore')
        except Exception: pass
        try:
            p = Path(path)
            if p.suffix.lower()=='.txt': return p.read_text()
        except Exception: pass
        return None
class TrackerService:
    def __init__(self, equipment_repo: 'EquipmentRepository', moves_repo: 'MovesLogRepository', sop_registry: 'SOPRegistry', qr: 'QRService', config):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG):
        return cls(
            EquipmentRepository(Path(_cfg(CONFIG,'EQUIPMENT_STATUS_PATH'))),
            MovesLogRepository(Path(_cfg(CONFIG,'EQUIPMENT_MOVES_LOG_PATH'))),
            SOPRegistry(Path(_cfg(CONFIG,'SOP_REGISTRY_PATH'))),
            QRService(Path(_cfg(CONFIG,'QR_OUTPUT_DIR'))), CONFIG)
    def equipment_status(self): return self.equipment_repo.read()
    def log_move(self, equip_id: str, loc_from: str, loc_to: str) -> None:
        ts = _utcnow_iso(); df = self.equipment_repo.read(); name = ''
        if 'name' in df.columns and (df['equip_id'].astype(str)==str(equip_id)).any():
            name = df.loc[df['equip_id'].astype(str)==str(equip_id), 'name'].iloc[0]
        rec = {'equip_id': str(equip_id), 'name': name, 'location': loc_to, 'status': 'moved', 'last_seen': ts, 'battery': np.nan, 'confidence': np.nan}
        self.equipment_repo.upsert(rec); self.moves_repo.append(str(equip_id), loc_from or '', loc_to, ts)
    def moves_summary(self):
        log = self.moves_repo.read()
        if log.empty: return {'moves_per_equipment': log, 'routes': log}
        per_eq = log.groupby('equip_id').size().reset_index(name='moves').sort_values('moves', ascending=False)
        routes = log.groupby(['from','to']).size().reset_index(name='count').sort_values('count', ascending=False)
        return {'moves_per_equipment': per_eq, 'routes': routes}
    def sop_table(self): return self.sop_registry.read()
    def search_sop(self, query: str):
        df = self.sop_registry.read().copy(); q = (query or '').strip().lower()
        if df is None or df.empty or not q: return df
        cols = [c for c in ['sop_id','title','keywords','version','status'] if c in df.columns]
        mask = df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
import types as _types, sys as _sys
_tracker_mod = _types.ModuleType('tracker_core')
for _name,_obj in {
    'EquipmentRepository': EquipmentRepository,
    'MovesLogRepository': MovesLogRepository,
    'SOPRegistry': SOPRegistry,
    'QRService': QRService,
    'TrackerService': TrackerService,
}.items(): setattr(_tracker_mod,_name,_obj)
_sys.modules['tracker_core'] = _tracker_mod
print('OK: tracker_core (v6 schema) ready.')


In [ ]:
CONFIG['RUN_UI']=False; CONFIG['RUN_PIPELINE']=False
print('OK: CONFIG flags set (RUN_UI=False, RUN_PIPELINE=False)')


In [ ]:
import numpy as np, pandas as pd, os
from pathlib import Path
from tracker_core import TrackerService, QRService, EquipmentRepository, MovesLogRepository, SOPRegistry
if not PHASE1_GATE:
    raise SystemExit('HARD GATE: canonical module missing or incomplete. Upload ed_pipeline_v8.py and re-run.')
os.environ['PYTHONHASHSEED']='0'; np.random.seed(0)
s=WorkflowState(role='nurse')
getattr(s,'touch_now',lambda *_:None)(pd.Timestamp.utcnow())
tc=TinyCritics(); p,b,u=tc.score(s,[{'id':'reassess_vitals','label':'Reassess vitals'},{'id':'order_ecg','label':'Order ECG'}])
assert len(p)==2 and (0<=p).all() and (p<=1).all(), 'TinyCritics bounds failed'
for key in ('EQUIPMENT_STATUS_PATH','EQUIPMENT_MOVES_LOG_PATH','SOP_REGISTRY_PATH','QR_OUTPUT_DIR'):
    Path(CONFIG[key]).parent.mkdir(parents=True, exist_ok=True)
t=TrackerService.from_config(CONFIG)
_=t.equipment_status()
t.log_move('pump-001','A1','B2')
assert Path(CONFIG['EQUIPMENT_MOVES_LOG_PATH']).exists(), 'Moves log path missing'
status=t.equipment_status()
for c in ['equip_id','name','location','status','last_seen','battery','confidence']:
    assert c in status.columns, f'missing column {c}'
print('PHASE1_GATE=PASS')
PHASE1_GATE=True


In [ ]:
from typing import Any, Dict
def refresh_sop_registry(CONFIG: Any, base_url: str='https://sop-notaufnahme.de/sop/') -> Dict[str,Any]:
    try:
        import requests; from bs4 import BeautifulSoup
        return {'ok': True, 'note': 'delegated (network call not executed here)'}
    except Exception as e:
        return {'ok': False, 'reason': f'missing libs: {e}'}
def qr_scan_fallback(payload: str, tracker: 'TrackerService'):
    try:
        parts=dict(kv.split('=',1) for kv in payload.split('&') if '=' in kv)
        eq_id=parts.get('id') or parts.get('equip_id'); to_loc=parts.get('to')
        if eq_id and to_loc:
            tracker.log_move(eq_id, '', to_loc); return {'ok': True, 'equip_id': eq_id, 'to': to_loc}
        return {'ok': False, 'reason': 'missing equip_id/to'}
    except Exception as e:
        return {'ok': False, 'reason': str(e)}
ALERT_THRESHOLDS_MIN={'equipment_overdue':60,'lingering_patient':120}
print('Surfaces present.')


In [ ]:
if not PHASE1_GATE:
    raise SystemExit('HARD GATE failed: Phase 1 must pass before Phase 2.')
if CONFIG.get('RUN_PIPELINE'):
    print('Phase 2 enabled — place guarded assertions here.')
else:
    print("Phase 2 disabled (set CONFIG['RUN_PIPELINE']=True to enable).")
